In [5]:
import os
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

@dataclass
class Config:
    seed: int = 42
    data_dir: str = './data'
    target: str = 'control_success'
    id_cols: list[str] = field(default_factory=lambda: [
        'row_id', 
        'batter_id', 
        'pitcher_id'
    ])
    cat_cols: list[str] = field(default_factory=lambda: [
        'top_bottom', 
        'game_type', 
        'game_dayofweek', 
        'game_month', 
        'pitcher_hand', 
        'batter_hand', 
        'pitcher_team_id', 
        'batter_team_id', 
        'base_state', 
        'count_state', 
        'inning_group', 
        'score_state_pitcher', 
        'hand_matchup'
    ])

def brier_skill_score(y_true, y_pred):
    r = y_true.mean()
    brier = ((y_pred - y_true) ** 2).mean()
    baseline_brier = r * (1 - r)
    score = max(0, 100000 * (1 - brier / baseline_brier))
    print(f'Brier: {brier:.6f} | 기준선: {baseline_brier:.6f}')
    print(f'Validation Score: {score:.2f}')
    return score, brier, baseline_brier

In [6]:
def make_features(df, prior_rate, prior_strength=50.0):
    featured = df.copy()
    featured['season_offset'] = featured['season'].astype('int16') - 2019
    featured['count_state'] = featured['balls_before'].astype('Int64').astype(str) + '-' + featured['strikes_before'].astype('Int64').astype(str)
    featured['is_two_strikes'] = (featured['strikes_before'] >= 2).astype('int8')
    featured['is_three_balls'] = (featured['balls_before'] >= 3).astype('int8')
    featured['is_full_count'] = ((featured['balls_before'] == 3) & (featured['strikes_before'] == 2)).astype('int8')
    featured['count_pressure'] = (featured['balls_before'] + featured['strikes_before']).astype('int8')
    featured['inning_group'] = pd.cut(featured['inning'], bins=[0, 3, 6, 9, np.inf], labels=['1-3', '4-6', '7-9', '10+']).astype('string')
    diff = featured['score_diff_pitcher_team'].astype('float32')
    featured['score_diff_abs'] = diff.abs()
    featured['score_state_pitcher'] = np.select([diff < 0, diff > 0], ['trailing', 'leading'], default='tie')
    featured['is_close_game'] = (diff.abs() <= 1).astype('int8')
    featured['has_risp'] = ((featured['runner_on_2b'] == 1) | (featured['runner_on_3b'] == 1)).astype('int8')
    featured['li_log1p'] = np.log1p(featured['li'].clip(lower=0))
    featured['hand_matchup'] = featured['pitcher_hand'].astype('string') + '_' + featured['batter_hand'].astype('string')
    pitcher_rate = featured['asof_pitcher_success_rate']
    batter_rate = featured['asof_batter_success_rate']
    featured['pitcher_cold_start'] = ((featured['asof_pitcher_n'] == 0) | pitcher_rate.isna()).astype('int8')
    featured['batter_cold_start'] = ((featured['asof_batter_n'] == 0) | batter_rate.isna()).astype('int8')
    featured['pitcher_success_smoothed'] = (featured['asof_pitcher_n'] * pitcher_rate.fillna(prior_rate) + prior_strength * prior_rate) / (featured['asof_pitcher_n'] + prior_strength)
    featured['batter_success_smoothed'] = (featured['asof_batter_n'] * batter_rate.fillna(prior_rate) + prior_strength * prior_rate) / (featured['asof_batter_n'] + prior_strength)
    p_strength = featured["pitcher_success_smoothed"] - prior_rate

    for matchup in ["1_1", "1_2", "2_1", "2_2"]:
        featured[f"pitcher_success_x_hand_{matchup}"] = (p_strength * (featured["hand_matchup"] == matchup).astype("int8"))

    return featured      

In [3]:
config = Config()
full_train = pd.read_csv(os.path.join(config.data_dir, 'train.csv'))
train = full_train.loc[full_train.season < 2024].copy()
val = full_train.loc[full_train.season == 2024].copy()

In [9]:
prior_rate = train[config.target].mean()
featured_train = make_features(train, prior_rate=prior_rate)
featured_val = make_features(val, prior_rate=prior_rate)

drop_cols = [
    'season', 
    'run_top_before', 
    'run_bot_before', 
    'score_diff_home', 
    'home_win_expectancy', 
    'away_win_expectancy',
    'runner_on_1b',
    'runner_on_2b',
    'runner_on_3b',
    'num_runners_on',
    'asof_pitcher_pitchmix_n',
    'top_bottom',
    'game_month',
    'game_dayofweek'
]

target = config.target
features = [col for col in featured_train.columns if col not in [target] + config.id_cols + drop_cols]
active_cat_cols = [col for col in config.cat_cols if col in features]

for frame in (featured_train, featured_val):
    for col in active_cat_cols:
        frame[col].astype("string").fillna("__MISSING__").astype(str)
        
X_train, y_train = featured_train[features], featured_train[target]
X_val, y_val = featured_val[features], featured_val[target]

In [13]:
base_params = {
    'loss_function': 'Logloss', 
    'eval_metric': 'BrierScore', 
    'iterations': 500, 
    'learning_rate': 0.05, 
    'l2_leaf_reg': 3, 
    'random_strength': 1, 
    'random_seed': config.seed, 
    'thread_count': -1, 
    'use_best_model': True, 
    'od_type': 'Iter', 
    'od_wait': 100, 
    'allow_writing_files': False, 
    'verbose': 100
}

model = CatBoostClassifier(**base_params)
model.fit(X_train, y_train, cat_features=active_cat_cols, eval_set=(X_val, y_val))
val_pred = model.predict_proba(X_val)[:, 1]
score, brier, baseline_brier = brier_skill_score(y_val, val_pred)

0:	learn: 0.2495647	test: 0.2499068	best: 0.2499068 (0)	total: 484ms	remaining: 4m 1s
100:	learn: 0.2445189	test: 0.2480052	best: 0.2480052 (100)	total: 42.7s	remaining: 2m 48s
200:	learn: 0.2441631	test: 0.2478898	best: 0.2478874 (195)	total: 1m 25s	remaining: 2m 6s
300:	learn: 0.2439025	test: 0.2478716	best: 0.2478597 (274)	total: 2m 7s	remaining: 1m 24s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2478596675
bestIteration = 274

Shrink model to first 275 iterations.
Brier: 0.247860 | 기준선: 0.249807
Validation Score: 779.51


HP tuning

In [29]:
base_params = {
    'loss_function': 'Logloss', 
    'eval_metric': 'BrierScore', 
    'iterations': 1000, 
    'learning_rate': 0.05, 
    'l2_leaf_reg': 3, 
    'random_strength': 1, 
    'random_seed': config.seed, 
    'thread_count': -1, 
    'use_best_model': True, 
    'od_type': 'Iter', 
    'od_wait': 150, 
    'allow_writing_files': False, 
    'verbose': 100
}

In [33]:
params = base_params.copy()
params.update({
            "grow_policy": "SymmetricTree",
            "depth": 7,
            "l2_leaf_reg": 1,
            "random_strength": 1,
            'bootstrap_type': 'MVS',
            'subsample':0.8
        })
model = CatBoostClassifier(**params)
model.fit(X_train,y_train,cat_features=active_cat_cols,eval_set=(X_val,y_val))

val_pred = model.predict_proba(X_val)[:, 1]
score, brier, baseline_brier = brier_skill_score(y_val,val_pred)

0:	learn: 0.2495571	test: 0.2499068	best: 0.2499068 (0)	total: 575ms	remaining: 9m 34s
100:	learn: 0.2442917	test: 0.2479138	best: 0.2479138 (100)	total: 50.5s	remaining: 7m 29s
200:	learn: 0.2438589	test: 0.2478385	best: 0.2478377 (178)	total: 1m 41s	remaining: 6m 41s
300:	learn: 0.2434813	test: 0.2478397	best: 0.2478257 (215)	total: 2m 31s	remaining: 5m 50s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.2478256818
bestIteration = 215

Shrink model to first 216 iterations.
Brier: 0.247826 | 기준선: 0.249807
Validation Score: 793.11


전체학습

In [7]:
import joblib

In [10]:
prior_rate = full_train[config.target].mean()
featured_train = make_features(full_train, prior_rate=prior_rate)

drop_cols = [
    'season', 
    'run_top_before', 
    'run_bot_before', 
    'score_diff_home', 
    'home_win_expectancy', 
    'away_win_expectancy',
    'runner_on_1b',
    'runner_on_2b',
    'runner_on_3b',
    'num_runners_on',
    'asof_pitcher_pitchmix_n',
    'top_bottom',
    'game_month',
    'game_dayofweek'
]

target = config.target
features = [col for col in featured_train.columns if col not in [target] + config.id_cols + drop_cols]
active_cat_cols = [col for col in config.cat_cols if col in features]

for col in active_cat_cols:
    featured_train[col].astype("string").fillna("__MISSING__").astype(str)
        
X_train, y_train = featured_train[features], featured_train[target]

final_params = {
    'loss_function': 'Logloss', 
    'eval_metric': 'BrierScore', 
    'iterations': 235, 
    'learning_rate': 0.05, 
    'random_seed': config.seed, 
    'thread_count': -1,  
    'allow_writing_files': False, 
    'verbose': 100,
    "grow_policy": "SymmetricTree",
    "depth": 7,
    "l2_leaf_reg": 1,
    "random_strength": 1,
    'bootstrap_type': 'MVS',
    'subsample':0.8
}

model = CatBoostClassifier(**final_params)
model.fit(X_train,y_train,cat_features=active_cat_cols)

os.makedirs("./model", exist_ok=True)
catboost_bundle = {
    "model": model,
    "drop_cols": drop_cols,
    "features": features,
    'prior_rate':prior_rate
}
joblib.dump(catboost_bundle, "./model/catboost_bundle.pkl", compress=3)
print("저장 완료: ./model/catboost_bundle.pkl")

0:	learn: 0.2496218	total: 668ms	remaining: 2m 36s
100:	learn: 0.2449586	total: 1m 1s	remaining: 1m 21s
200:	learn: 0.2445254	total: 1m 57s	remaining: 20s
234:	learn: 0.2444143	total: 2m 17s	remaining: 0us
저장 완료: ./model/catboost_bundle.pkl


prediction 분포 확인

In [103]:
from sklearn.linear_model import LogisticRegression
import numpy as np

def fit_platt_calibrator(raw_pred, y):
    raw_pred = np.clip(raw_pred, 1e-6, 1 - 1e-6)
    logit_pred = np.log(raw_pred / (1 - raw_pred))

    calibrator = LogisticRegression(C=1e6)
    calibrator.fit(logit_pred.reshape(-1, 1), y)

    return calibrator


def apply_platt_calibrator(calibrator, raw_pred):
    raw_pred = np.clip(raw_pred, 1e-6, 1 - 1e-6)
    logit_pred = np.log(raw_pred / (1 - raw_pred))

    return calibrator.predict_proba(logit_pred.reshape(-1, 1))[:, 1]

# 2024년을 calibration/evaluation으로 분리
cut = int(len(val) * 0.7)

calib_idx = np.arange(cut)
eval_idx = np.arange(cut, len(val))

raw_calib_pred = model.predict_proba(
    X_val.iloc[calib_idx]
)[:, 1]

raw_eval_pred = model.predict_proba(
    X_val.iloc[eval_idx]
)[:, 1]

calibrator = fit_platt_calibrator(
    raw_calib_pred,
    y_val.iloc[calib_idx]
)

calibrated_eval_pred = apply_platt_calibrator(
    calibrator,
    raw_eval_pred
)

from sklearn.metrics import brier_score_loss

raw_brier = brier_score_loss(
    y_val.iloc[eval_idx],
    raw_eval_pred
)

calibrated_brier = brier_score_loss(
    y_val.iloc[eval_idx],
    calibrated_eval_pred
)

raw_brier_skill,_,_ = brier_skill_score(
    y_val.iloc[eval_idx],
    raw_eval_pred
)

cal_brier_skill,_,_ = brier_skill_score(
    y_val.iloc[eval_idx],
    calibrated_eval_pred
)

print("Raw Brier:", raw_brier)
print("Calibrated Brier:", calibrated_brier)
print("Raw Brier Score:", f'{raw_brier_skill:.2f}')
print("Calibrated Brier Score:", f'{cal_brier_skill:.2f}')

Brier: 0.248404 | 기준선: 0.249640
Validation Score: 495.15
Brier: 0.248396 | 기준선: 0.249640
Validation Score: 498.37
Raw Brier: 0.24840415482868183
Calibrated Brier: 0.24839611467319295
Raw Brier Score: 495.15
Calibrated Brier Score: 498.37


In [9]:
analysis_val = featured_val.copy()

analysis_val["pred_prob"] = val_pred
analysis_val["error"] = analysis_val["pred_prob"] - analysis_val[config.target]
analysis_val["abs_error"] = analysis_val["error"].abs()
analysis_val["brier_contribution"] = analysis_val["error"] ** 2

worst_samples = (
    analysis_val
    .sort_values("brier_contribution", ascending=False)
    [[
        "row_id",
        "season",
        "pitcher_id",
        "batter_id",
        "pred_prob",
        config.target,
        "abs_error",
        "balls_before",
        "strikes_before",
        "outs_before",
        "base_state",
        "inning",
        "score_diff_pitcher_team",
        "pitcher_hand",
        "batter_hand",
    ]]
    .head(30)
)

worst_samples

,row_id,season,pitcher_id,batter_id,pred_prob,control_success,abs_error,balls_before,strikes_before,outs_before,base_state,inning,score_diff_pitcher_team,pitcher_hand,batter_hand
1348124,TRAIN_1348125,2024,23903,22410,0.340908,1,0.659092,3,2,1,___,6,1,1,1
1348123,TRAIN_1348124,2024,23903,22410,0.340908,1,0.659092,3,2,1,___,6,1,1,1
1422102,TRAIN_1422103,2024,23903,24548,0.341391,1,0.658609,3,0,2,___,5,-6,1,1
1405565,TRAIN_1405566,2024,23903,22736,0.342721,1,0.657279,3,2,2,___,1,1,1,1
1405644,TRAIN_1405645,2024,23903,23281,0.346031,1,0.653969,3,1,0,___,4,0,1,1
1256222,TRAIN_1256223,2024,24488,23578,0.346116,1,0.653884,3,1,0,___,5,1,1,1
1265735,TRAIN_1265736,2024,24012,23810,0.346350,1,0.653650,0,1,0,___,2,-1,1,1
1294270,TRAIN_1294271,2024,23903,22410,0.347100,1,0.652900,3,2,0,___,6,4,1,1
1316775,TRAIN_1316776,2024,24115,22430,0.347137,1,0.652863,3,2,1,1_3,2,2,1,1
1342991,TRAIN_1342992,2024,24544,24500,0.347449,1,0.652551,3,1,2,___,4,-3,1,1


In [10]:
overconfident_failure = (
    analysis_val[
        (analysis_val[config.target] == 0)
        & (analysis_val["pred_prob"] >= 0.8)
    ]
    .sort_values("pred_prob", ascending=False)
)

overconfident_failure

,row_id,season,game_month,game_dayofweek,inning,top_bottom,game_type,balls_before,strikes_before,outs_before,...,pitcher_recent_form_gap,pitcher_recent_vs_baseline,pitcher_medium_vs_baseline,pitcher_long_vs_baseline,pitcher_n_log1p,batter_n_log1p,pred_prob,error,abs_error,brier_contribution


In [11]:
underestimated_success = (
    analysis_val[
        (analysis_val[config.target] == 1)
        & (analysis_val["pred_prob"] <= 0.2)
    ]
    .sort_values("pred_prob")
)

underestimated_success

,row_id,season,game_month,game_dayofweek,inning,top_bottom,game_type,balls_before,strikes_before,outs_before,...,pitcher_recent_form_gap,pitcher_recent_vs_baseline,pitcher_medium_vs_baseline,pitcher_long_vs_baseline,pitcher_n_log1p,batter_n_log1p,pred_prob,error,abs_error,brier_contribution


In [12]:
analysis_val["prob_bin"] = pd.cut(
    analysis_val["pred_prob"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

calibration_table = (
    analysis_val
    .groupby("prob_bin", observed=False)
    .agg(
        n=(config.target, "size"),
        mean_pred=("pred_prob", "mean"),
        actual_rate=(config.target, "mean"),
        brier=("brier_contribution", "mean"),
    )
    .reset_index()
)

calibration_table["calibration_gap"] = (
    calibration_table["actual_rate"]
    - calibration_table["mean_pred"]
)

calibration_table

,prob_bin,n,mean_pred,actual_rate,brier,calibration_gap
0,"(-0.001, 0.1]",0,NaN,NaN,NaN,NaN
1,"(0.1, 0.2]",0,NaN,NaN,NaN,NaN
2,"(0.2, 0.3]",0,NaN,NaN,NaN,NaN
3,"(0.3, 0.4]",3107,0.387855,0.383328,0.236465,-0.004527
4,"(0.4, 0.5]",144410,0.466723,0.458916,0.247824,-0.007808
5,"(0.5, 0.6]",103404,0.532511,0.524071,0.248514,-0.008440
6,"(0.6, 0.7]",2586,0.610431,0.609822,0.237633,-0.000609
7,"(0.7, 0.8]",0,NaN,NaN,NaN,NaN
8,"(0.8, 0.9]",0,NaN,NaN,NaN,NaN
9,"(0.9, 1.0]",0,NaN,NaN,NaN,NaN


In [13]:
def group_brier(df, group_col):
    result = (
        df.groupby(group_col, dropna=False)
        .agg(
            n=(config.target, "size"),
            actual_rate=(config.target, "mean"),
            mean_pred=("pred_prob", "mean"),
            brier=("brier_contribution", "mean"),
        )
        .reset_index()
    )

    result["calibration_gap"] = (
        result["actual_rate"] - result["mean_pred"]
    )

    return result.sort_values("brier", ascending=False)

In [14]:
group_brier(analysis_val, "count_state")
group_brier(analysis_val, "base_state")
group_brier(analysis_val, "inning_group")
group_brier(analysis_val, "hand_matchup")
group_brier(analysis_val, "score_state_pitcher")
group_brier(analysis_val, "pitcher_hand")
group_brier(analysis_val, "batter_hand")

,batter_hand,n,actual_rate,mean_pred,brier,calibration_gap
0,1,119761,0.48687,0.495927,0.247883,-0.009058
1,2,133746,0.48542,0.492382,0.247843,-0.006962


In [32]:
analysis_val = featured_val.copy()
analysis_val["pred_prob"] = val_pred
analysis_val["error"] = (
    analysis_val["pred_prob"] - analysis_val[config.target]
)
analysis_val["abs_error"] = analysis_val["error"].abs()
analysis_val["brier_contribution"] = analysis_val["error"] ** 2

In [15]:
pitcher_error = (
    analysis_val
    .groupby("pitcher_id")
    .agg(
        n=(config.target, "size"),
        actual_rate=(config.target, "mean"),
        mean_pred=("pred_prob", "mean"),
        brier=("brier_contribution", "mean"),
        mae=("abs_error", "mean"),
    )
    .query("n >= 100")
    .sort_values("brier", ascending=False)
)

pitcher_error.head(30)

,n,actual_rate,mean_pred,brier,mae
pitcher_id,,,,,
23930,147,0.489796,0.455666,0.255072,0.502585
23867,199,0.502513,0.440713,0.255035,0.501190
22717,162,0.512346,0.451039,0.254187,0.500622
23597,564,0.491135,0.478344,0.253778,0.502102
23515,105,0.523810,0.464557,0.253420,0.501656
23439,242,0.446281,0.521071,0.253184,0.502459
24166,270,0.559259,0.484452,0.253123,0.502683
23761,284,0.485915,0.454191,0.252798,0.499903
24596,125,0.544000,0.479434,0.252779,0.501986
